In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval pandas pillow
except Exception:
    pass

In [2]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from dataeval import Metadata
from dataeval.protocols import AnnotatedDataset, DatasetMetadata, DatumMetadata

In [3]:
data_dir = Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)

index2label = {0: "cat", 1: "dog", 2: "bird"}
weather_options = ["clear", "rainy", "foggy"]

rows = []
# Creating 90 128x128 images
for i in range(90):
    # Stand-in for a real image file - replace with your own images on disk
    pixels = rng.integers(0, 256, size=(128, 128, 3), dtype=np.uint8)
    filepath = data_dir / f"img_{i:03d}.png"
    Image.fromarray(pixels).save(filepath)

    rows.append({
        "filepath": str(filepath),
        "label": i % 3,
        "weather": weather_options[(i // 3) % 3],
        "altitude_m": float(50 + i),
    })

catalog = pd.DataFrame(rows)
catalog.head()

,filepath,label,weather,altitude_m
0,/tmp/tmpj8c5kzds/img_000.png,0,clear,50.0
1,/tmp/tmpj8c5kzds/img_001.png,1,clear,51.0
2,/tmp/tmpj8c5kzds/img_002.png,2,clear,52.0
3,/tmp/tmpj8c5kzds/img_003.png,0,rainy,53.0
4,/tmp/tmpj8c5kzds/img_004.png,1,rainy,54.0


In [4]:
class DataFrameICDataset:
    """An image classification dataset backed by a pandas DataFrame.

    Each row describes one image via an image-path column, a label column, and any
    number of metadata columns that are surfaced to DataEval as datum metadata.
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        index2label: dict[int, str],
        image_col: str = "filepath",
        label_col: str = "label",
        metadata_cols: list[str] | None = None,
        dataset_id: str = "dataframe-catalog",
    ) -> None:
        # reset_index keeps positional indexing aligned with __getitem__
        self._df = dataframe.reset_index(drop=True)
        self.index2label = index2label
        self._image_col = image_col
        self._label_col = label_col
        self._metadata_cols = metadata_cols or []
        # DatasetMetadata advertises the class-name mapping to DataEval
        self.metadata: DatasetMetadata = DatasetMetadata(id=dataset_id, index2label=index2label)

    def __len__(self) -> int:
        return len(self._df)

    def _one_hot(self, label: int) -> np.ndarray:
        target = np.zeros(len(self.index2label), dtype=np.float32)
        target[label] = 1.0
        return target

    def __getitem__(self, index: int) -> tuple[np.ndarray, np.ndarray, DatumMetadata]:
        row = self._df.iloc[index]

        # Decode the image and convert to channels-first (C, H, W)
        image = np.asarray(Image.open(row[self._image_col]).convert("RGB"), dtype=np.uint8).transpose(2, 0, 1)

        # One-hot encode the label as the classification target
        target = self._one_hot(int(row[self._label_col]))

        # Pull in the desired metadata columns; remember metadata is per-image and must include an "id" field
        datum_metadata = DatumMetadata(id=index, **{col: row[col] for col in self._metadata_cols})

        return image, target, datum_metadata

In [5]:
dataset = DataFrameICDataset(catalog, index2label, metadata_cols=["weather", "altitude_m"])
print(f"Dataset length: {len(dataset)}")

Dataset length: 90


In [6]:
image, target, datum_metadata = dataset[0]
print(f"image shape:    {image.shape} ({image.dtype})")
print(f"target (label): {target}")
print(f"datum metadata: {datum_metadata}")

image shape:    (3, 128, 128) (uint8)
target (label): [1. 0. 0.]
datum metadata: {'id': 0, 'weather': 'clear', 'altitude_m': np.float64(50.0)}


In [7]:
print(f"Is an AnnotatedDataset: {isinstance(dataset, AnnotatedDataset)}")

Is an AnnotatedDataset: True


In [8]:
metadata = Metadata(dataset)

# "id" is a per-item identifier, not a meaningful factor for bias analysis
metadata.exclude = ["id"]
print(f"Factor names: {metadata.factor_names}")

Factor names: ['altitude_m', 'weather']
